In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone -b feat/data-augm --single-branch https://github.com/deadPixelsGreta/xAI-proj-m-ws2526.git

In [ ]:
import shutil
import os
from tqdm import tqdm

source_path = "/content/drive/MyDrive/ImageNetSubset/"
destination_path = "/content/xAI-proj-m-ws2526/datasets/"

# If the destination directory exists, remove it first
if os.path.exists(destination_path):
    print(f"Removing existing directory: {destination_path}")
    shutil.rmtree(destination_path)

# Custom copy function with tqdm
def copytree_with_tqdm(src, dst):
    # Calculate total number of items (files and directories) to copy for tqdm
    total_items = 0
    for dirpath, dirnames, filenames in os.walk(src):
        total_items += len(dirnames) # for directories
        total_items += len(filenames) # for files

    # Ensure the destination root directory exists
    os.makedirs(dst, exist_ok=True)

    with tqdm(total=total_items, unit="item", desc=f"Copying {os.path.basename(src)}") as pbar:
        for dirpath, dirnames, filenames in os.walk(src):
            # Create subdirectories in destination
            relative_path = os.path.relpath(dirpath, src)
            current_dst_dir = os.path.join(dst, relative_path)

            for dirname in dirnames:
                dest_dir = os.path.join(current_dst_dir, dirname)
                os.makedirs(dest_dir, exist_ok=True)
                pbar.update(1)

            # Copy files
            for filename in filenames:
                src_file = os.path.join(dirpath, filename)
                dst_file = os.path.join(current_dst_dir, filename)
                shutil.copy2(src_file, dst_file)
                pbar.update(1)

# Call the custom copy function
copytree_with_tqdm(source_path, destination_path)

In [ ]:
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    """Walk upward to find the outermost folder containing common project markers."""
    markers = {".git", "requirements.txt", "setup.py", "pyproject.toml"}
    root = None
    for parent in [start, *start.parents]:
        if any((parent / m).exists() for m in markers):
            root = parent  # keep going to prefer the outermost match
    return root or start

# Dynamically get the name of the cloned repository if it exists
cloned_repo_name = "xAI-proj-m-ws2526"
cloned_repo_path = Path.cwd() / cloned_repo_name

# If the cloned repository exists as a subdirectory, change into it
if cloned_repo_path.is_dir():
    os.chdir(cloned_repo_path)

# Now, find the project root from within the repository (or its parent if already there)
ROOT = find_project_root(Path.cwd()).resolve()

# Ensure we are in the identified project root
os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))
print("cwd:", Path.cwd())
print("root on sys.path:", str(ROOT) in sys.path)

In [ ]:
# 2. Install dependencies
!pip install -r experiments/requirements.txt --quiet

In [ ]:
# 5. Train ResNet-18 (via config)
!python -m experiments.scripts.train --config experiments/configs/default.yaml --model resnet18

In [ ]:
# 6. Train ResNet-34 (via config)
!python -m experiments.scripts.train --config experiments/configs/default.yaml --model resnet34

In [ ]:
# 7. Train EfficientNet-B0 (via config)
!python -m experiments.scripts.train --config experiments/configs/default.yaml --model efficientnet_b0

---

## Validation of Ensemble

In [ ]:
# 4. Run ensemble evaluation on a dataset
!python -m experiments.scripts.inference --evaluate --data-dir datasets

In [ ]:
# 3. Upload a test image or use sample
!python -m experiments.scripts.inference --image datasets/test_image.jpg --show-individual